In [0]:
# 01_landing_to_bronze.py
s3_ext_location = 's3://awae-yellow-taxi-case'

from pyspark.sql.functions import col, split, element_at
from pyspark.sql.types import DoubleType

In [0]:
# Reusable function to process individual monthly files, standardize schema, and save to Bronze (Delta)
def process_taxi_data_to_bronze(s3_base_path: str, taxi_type: str, months: list):
    print(f"--- Starting Bronze Processing for {taxi_type.upper()} Taxi Data ---")
    
    processed_dfs = []
    for month in months:
        file_path = f"{s3_base_path}/landing/{taxi_type}_tripdata/{taxi_type}_tripdata_2023-{month}.parquet"
        print(f"Reading individual file: {file_path}")
        
        # Read each file individually to safely map and intercept specific headers
        df = spark.read.format("parquet").load(file_path)
        
        # Standardize column types for passenger_count and RatecodeID across all files
        df = df.withColumn("passenger_count", col("passenger_count").cast(DoubleType())) \
               .withColumn("RatecodeID", col("RatecodeID").cast(DoubleType()))
        
        # Handle case inconsistency in airport_fee (specifically found in Yellow Taxi files)
        if "Airport_fee" in df.columns:
            df = df.withColumnRenamed("Airport_fee", "airport_fee")
            
        # Capture the source file name using Unity Catalog native _metadata column
        df = df.withColumn("source_file", element_at(split(col("_metadata.file_path"), "/"), -1))
        
        processed_dfs.append(df)

    # Consolidate all individual monthly dataframes using unionByName
    print(f"Consolidating monthly dataframes for {taxi_type} taxi...")
    df_consolidated = processed_dfs[0]
    for df in processed_dfs[1:]:
        df_consolidated = df_consolidated.unionByName(df)

    # Save the clean data stream to its specific Bronze Delta location
    path_bronze = f"{s3_base_path}/bronze/{taxi_type}_tripdata_bronze"
    print(f"Saving consolidated data to Bronze layer (Delta): {path_bronze}")
    df_consolidated.write.format("delta").mode("overwrite").save(path_bronze)
    print(f"{taxi_type.upper()} Taxi Bronze layer successfully updated!\n")

In [0]:
# Execution baseline variables
target_months = ["01", "02", "03", "04", "05"]

# Run the modularized pipeline for both sources sequentially
process_taxi_data_to_bronze(s3_base_path=s3_ext_location, taxi_type="yellow", months=target_months)
process_taxi_data_to_bronze(s3_base_path=s3_ext_location, taxi_type="green", months=target_months)

print("Bronze layer successfully updated for all sources using Databricks Serverless!")